In [ ]:
# import numpy as np
# import pandas as pd

# def _parse_flat_values(raw_text, expected_count):
#     # First try splitting on whitespace
#     vals = np.fromstring(raw_text, sep=' ')
#     if vals.size == expected_count:
#         return vals

#     # Fallback: remove newlines, keep all characters
#     compact = raw_text.replace('\n', '')
#     # Heuristic: estimate field width
#     if len(compact) % expected_count != 0:
#         raise ValueError(f"Cannot infer fixed width: total chars {len(compact)} not divisible by expected {expected_count}")
#     width = len(compact) // expected_count
#     parts = []
#     for i in range(expected_count):
#         chunk = compact[i*width:(i+1)*width]
#         try:
#             parts.append(float(chunk))
#         except ValueError:
#             # try stripping before conversion
#             try:
#                 parts.append(float(chunk.strip()))
#             except ValueError:
#                 raise ValueError(f"Failed to parse chunk '{chunk}' as float in fixed-width fallback")
#     return np.array(parts)

# def parse_simulps_velocity_model(filepath):
#     with open(filepath, 'r') as f:
#         lines = f.readlines()

#     # === Header ===
#     header = lines[0].split()
#     nx, ny, nz = int(header[1]), int(header[2]), int(header[3])
#     expected = nx * ny * nz

#     # === Coordinate grids ===
#     x_vals = np.fromstring(lines[1], sep=' ')
#     y_vals = np.fromstring(lines[2], sep=' ')
#     z_vals = np.fromstring(lines[3], sep=' ')

#     # === Data lines (skip dummy line at index 4) ===
#     data_lines = lines[5:]
#     raw_data = ''.join(data_lines)

#     # Try to parse all numeric values flexibly
#     all_values = _parse_flat_values(raw_data, expected * 3)  # max possible (Vp, Vs, Vp/Vs)

#     # Determine how many model volumes are actually present
#     if all_values.size == expected:
#         nmodels = 1
#     elif all_values.size == 2 * expected:
#         nmodels = 2
#     elif all_values.size == 3 * expected:
#         nmodels = 3
#     else:
#         raise ValueError(f"Unexpected number of data values: got {all_values.size}, expected 1x,2x or3x {expected}")

#     volumes = []
#     for i in range(nmodels):
#         start = i * expected
#         end = (i + 1) * expected
#         vol_flat = all_values[start:end]
#         vol = vol_flat.reshape((nz, ny, nx))  # [z, y, x]
#         vol = np.transpose(vol, (2, 1, 0))    # -> [x, y, z]
#         volumes.append(vol)

#     vp = volumes[0]
#     vs = volumes[1] if nmodels >= 2 else None
#     vpratio = volumes[2] if nmodels >= 3 else None

#     return {
#         "x": x_vals,
#         "y": y_vals,
#         "z": z_vals,
#         "vp": vp,
#         "vs": vs,
#         "vp_vs_ratio": vpratio
#     }


# datadir = "/home/staff/chao/SSEinv/Nicoya/DeShon_2006GJI/"
# modelFile = "NICOYA_MODEL/eq10ps.f3dps.overdamp.mod.out"
# model = parse_simulps_velocity_model(datadir + modelFile)

# x, y, z = model["x"], model["y"], model["z"]
# vp = model["vp"]            # always present
# vs = model["vs"]            # may be None if not in file
# vpratio = model["vp_vs_ratio"]  # may be None

# print("vp shape:", vp.shape)  # (nx, ny, nz)
# if vs is not None:
#     print("vs shape:", vs.shape)
# if vpratio is not None:
#     print("Vp/Vs shape:", vpratio.shape)

In [ ]:
# import numpy as np
# import re

# def detect_simulps_content(filename):
#     """
#     Detect what velocity data is contained in a SIMULPS file based on line count.
    
#     Parameters:
#     -----------
#     filename : str
#         Path to the SIMULPS velocity file
        
#     Returns:
#     --------
#     dict containing:
#         - 'content_type': str - 'vp_only', 'vp_vs', or 'vp_vs_ratio'
#         - 'has_vp': bool
#         - 'has_vs': bool  
#         - 'has_vp_vs_ratio': bool
#         - 'expected_data_lines': int
#         - 'actual_data_lines': int
#         - 'nx': int, 'ny': int, 'nz': int
#     """
    
#     with open(filename, 'r') as f:
#         lines = f.readlines()
    
#     # Parse header to get dimensions
#     header = lines[0].strip().split()
#     nx = int(header[1])
#     ny = int(header[2]) 
#     nz = int(header[3])
    
#     # Calculate expected lines
#     header_lines = 5  # header + 3 coordinate lines + "0 0 0" line
#     total_lines = len(lines)
#     data_lines = total_lines - header_lines
    
#     lines_per_layer = nx  # Each depth layer has nx rows
#     lines_for_one_dataset = lines_per_layer * nz  # One complete 3D dataset
    
#     print(f"File analysis:")
#     print(f"Grid dimensions: {nx} x {ny} x {nz}")
#     print(f"Total lines in file: {total_lines}")
#     print(f"Header lines: {header_lines}")
#     print(f"Data lines: {data_lines}")
#     print(f"Lines per complete dataset: {lines_for_one_dataset}")
    
#     # Determine content based on data lines
#     if data_lines == lines_for_one_dataset:
#         content_type = 'vp_only'
#         has_vp, has_vs, has_ratio = True, False, False
#         print("Content detected: Vp only")
        
#     elif data_lines == 2 * lines_for_one_dataset:
#         content_type = 'vp_vs'
#         has_vp, has_vs, has_ratio = True, True, False
#         print("Content detected: Vp and Vs")
        
#     elif data_lines == 3 * lines_for_one_dataset:
#         content_type = 'vp_vs_ratio'
#         has_vp, has_vs, has_ratio = True, True, True
#         print("Content detected: Vp, Vs, and Vp/Vs ratio")
        
#     else:
#         print(f"WARNING: Unexpected number of data lines!")
#         print(f"Expected: {lines_for_one_dataset} (Vp only), {2*lines_for_one_dataset} (Vp+Vs), or {3*lines_for_one_dataset} (Vp+Vs+ratio)")
#         print(f"Found: {data_lines}")
#         content_type = 'unknown'
#         has_vp, has_vs, has_ratio = False, False, False
    
#     return {
#         'content_type': content_type,
#         'has_vp': has_vp,
#         'has_vs': has_vs,
#         'has_vp_vs_ratio': has_ratio,
#         'expected_data_lines': lines_for_one_dataset,
#         'actual_data_lines': data_lines,
#         'nx': nx,
#         'ny': ny,
#         'nz': nz
#     }

# def parse_simulps_velocity_file(filename):
#     """
#     Parse a SIMULPS velocity model file and extract grids and velocity data.
#     Automatically detects whether file contains Vp only, Vp+Vs, or Vp+Vs+ratio.
    
#     Parameters:
#     -----------
#     filename : str
#         Path to the SIMULPS velocity file
        
#     Returns:
#     --------
#     dict containing:
#         - 'x_grid': numpy array of x coordinates
#         - 'y_grid': numpy array of y coordinates  
#         - 'z_grid': numpy array of z coordinates (depths)
#         - 'vp': numpy array of shape (nx, ny, nz) with P-wave velocity values (if present)
#         - 'vs': numpy array of shape (nx, ny, nz) with S-wave velocity values (if present)
#         - 'vp_vs_ratio': numpy array of shape (nx, ny, nz) with Vp/Vs ratio values (if present)
#         - 'metadata': dict with file metadata
#         - 'content_info': dict with content detection results
#     """
    
#     # First detect what's in the file
#     content_info = detect_simulps_content(filename)
#     print(content_info['content_type'])  # 'vp_only', 'vp_vs', or 'vp_vs_ratio'
    
#     with open(filename, 'r') as f:
#         lines = f.readlines()

#     # Parse only the first 4 fields
#     header = lines[0].strip().split()[:4]
#     version = float(header[0])
#     nx = int(header[1])
#     ny = int(header[2])
#     nz = int(header[3])

#     print(f"Parsing SIMULPS file: {nx}x{ny}x{nz} grid (version {version})")
        
#     # Parse coordinate arrays
#     x_coords = np.array([float(x) for x in lines[1].strip().split()])
#     y_coords = np.array([float(y) for y in lines[2].strip().split()])
#     z_coords = np.array([float(z) for z in lines[3].strip().split()])
    
#     # Verify coordinate array lengths
#     if len(x_coords) != nx:
#         raise ValueError(f"X coordinates length ({len(x_coords)}) doesn't match nx ({nx})")
#     if len(y_coords) != ny:
#         raise ValueError(f"Y coordinates length ({len(y_coords)}) doesn't match ny ({ny})")
#     if len(z_coords) != nz:
#         raise ValueError(f"Z coordinates length ({len(z_coords)}) doesn't match nz ({nz})")
    
#     # Skip the "0 0 0" line (line 4, index 3)
#     data_start_line = 5
    
#     # Initialize velocity arrays for Vp, Vs, and Vp/Vs ratio
#     vp = np.zeros((nx, ny, nz))
#     vs = np.zeros((nx, ny, nz))
#     vp_vs_ratio = np.zeros((nx, ny, nz))
    
#     # Parse velocity data based on detected content
#     current_line = data_start_line
    
#     # Parse Vp data (always present)
#     if content_info['has_vp']:
#         print("Reading Vp data...")
#         for k in range(nz):  # For each depth layer
#             print(f"  Vp depth layer {k+1}/{nz} (z = {z_coords[k]} km)")
            
#             # Read nx rows for this depth layer
#             for i in range(nx):  # For each x row
#                 if current_line >= len(lines):
#                     raise ValueError(f"Unexpected end of file at Vp depth layer {k+1}, row {i+1}")
                    
#                 line = lines[current_line].strip()
#                 current_line += 1
                
#                 # Parse the line
#                 row_values = parse_velocity_line(line, ny)
                
#                 if len(row_values) != ny:
#                     raise ValueError(f"Vp row {i+1} in depth layer {k+1} has {len(row_values)} values, expected {ny}")
                
#                 # File row i (0-indexed) corresponds to x_coords[i]
#                 # File column j (0-indexed) corresponds to y_coords[ny-1-j] (reversed Y)
#                 # So: vp[i, ny-1-j, k] = row_values[j]
#                 for j in range(ny):
#                     vp[i, ny-1-j, k] = row_values[j]
    
#     # Parse Vs data (if present)
#     if content_info['has_vs']:
#         print("Reading Vs data...")
#         for k in range(nz):  # For each depth layer
#             print(f"  Vs depth layer {k+1}/{nz} (z = {z_coords[k]} km)")
            
#             # Read nx rows for this depth layer
#             for i in range(nx):  # For each x row
#                 if current_line >= len(lines):
#                     raise ValueError(f"Unexpected end of file at Vs depth layer {k+1}, row {i+1}")
                    
#                 line = lines[current_line].strip()
#                 current_line += 1
                
#                 # Parse the line
#                 row_values = parse_velocity_line(line, ny)
                
#                 if len(row_values) != ny:
#                     raise ValueError(f"Vs row {i+1} in depth layer {k+1} has {len(row_values)} values, expected {ny}")
                
#                 # File row i (0-indexed) corresponds to x_coords[i]
#                 # File column j (0-indexed) corresponds to y_coords[ny-1-j] (reversed Y)
#                 # So: vs[i, ny-1-j, k] = row_values[j]
#                 for j in range(ny):
#                     vs[i, ny-1-j, k] = row_values[j]
    
#     # Parse Vp/Vs ratio data (if present)
#     if content_info['has_vp_vs_ratio']:
#         print("Reading Vp/Vs ratio data...")
#         for k in range(nz):  # For each depth layer
#             print(f"  Vp/Vs depth layer {k+1}/{nz} (z = {z_coords[k]} km)")
            
#             # Read nx rows for this depth layer
#             for i in range(nx):  # For each x row
#                 if current_line >= len(lines):
#                     raise ValueError(f"Unexpected end of file at Vp/Vs depth layer {k+1}, row {i+1}")
                    
#                 line = lines[current_line].strip()
#                 current_line += 1
                
#                 # Parse the line
#                 row_values = parse_velocity_line(line, ny)
                
#                 if len(row_values) != ny:
#                     raise ValueError(f"Vp/Vs row {i+1} in depth layer {k+1} has {len(row_values)} values, expected {ny}")
                
#                 # File row i (0-indexed) corresponds to x_coords[i]
#                 # File column j (0-indexed) corresponds to y_coords[ny-1-j] (reversed Y)
#                 # So: vp_vs_ratio[i, ny-1-j, k] = row_values[j]
#                 for j in range(ny):
#                     vp_vs_ratio[i, ny-1-j, k] = row_values[j]
    
#     # Create metadata dictionary
#     metadata = {
#         'version': version,
#         'nx': nx,
#         'ny': ny, 
#         'nz': nz,
#     }
    
#     return {
#         'x_grid': x_coords,
#         'y_grid': y_coords,
#         'z_grid': z_coords,
#         'vp': vp,
#         'vs': vs,
#         'vp_vs_ratio': vp_vs_ratio,
#         'metadata': metadata,
#         'content_info': content_info
#     }


# def parse_velocity_line(line, expected_count):
#     """
#     Parse a line of velocity values that may be space-separated or concatenated.
    
#     Parameters:
#     -----------
#     line : str
#         Line containing velocity values
#     expected_count : int
#         Expected number of values in the line
        
#     Returns:
#     --------
#     list of float values
#     """
    
#     # First try space-separated parsing
#     if ' ' in line:
#         values = [float(x) for x in line.split()]
#         if len(values) == expected_count:
#             return values
    
#     # If space-separated doesn't work or gives wrong count, try pattern matching
#     # Look for patterns like "3.00", "10.30", etc.
    
#     # Try different decimal patterns
#     patterns = [
#         r'\d+\.\d{2}',  # Format like 3.00, 10.30
#         r'\d+\.\d{1}',  # Format like 3.0, 10.3  
#         r'\d+\.\d{3}',  # Format like 3.000, 10.300
#         r'\d+\.\d+',    # Any decimal format
#         r'\d+'          # Integer format
#     ]
    
#     for pattern in patterns:
#         matches = re.findall(pattern, line)
#         if len(matches) == expected_count:
#             return [float(x) for x in matches]
    
#     # If pattern matching fails, try fixed-width parsing
#     # Assume each value takes the same number of characters
#     line_clean = line.replace(' ', '')
#     if len(line_clean) % expected_count == 0:
#         width = len(line_clean) // expected_count
#         values = []
#         for i in range(expected_count):
#             start_idx = i * width
#             end_idx = start_idx + width
#             value_str = line_clean[start_idx:end_idx]
#             try:
#                 values.append(float(value_str))
#             except ValueError:
#                 break
#         if len(values) == expected_count:
#             return values
    
#     # If all else fails, raise an error
#     raise ValueError(f"Could not parse velocity line: '{line}' (expected {expected_count} values)")

# def export_to_csv(data, output_filename):
#     """
#     Export the parsed velocity model to a CSV file with columns: x, y, z, vp, vs, vp_vs_ratio
    
#     Parameters:
#     -----------
#     data : dict
#         Dictionary returned by parse_simulps_velocity_file
#     output_filename : str
#         Output CSV filename
#     """
    
#     import csv
#     import utils as ut
    
#     nx, ny, nz = data['metadata']['nx'], data['metadata']['ny'], data['metadata']['nz']
    
#     print(f"Exporting velocity model to CSV: {output_filename}")
#     print(f"Grid dimensions: {nx} x {ny} x {nz} = {nx*ny*nz} points")
    
#     with open(output_filename, 'w', newline='') as csvfile:
#         fieldnames = ['lon', 'lat', 'x', 'y', 'z', 'vp', 'vs', 'vp_vs_ratio']
#         writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
#         # Write header
#         writer.writeheader()
        
#         # define the centroid of relative coordinates,
#         # found from the fist line in "DeShon_2006GJI/NICOYA_MODEL/eq10ps.f3dps.overdamp.stns.out"
#         lon0, lat0 = -(84+42.00/60), 9+12.00/60 

#         # Write data for each grid point
#         for i in range(nx):
#             for j in range(ny):
#                 for k in range(nz):
#                     # Get coordinates
#                     x = data['x_grid'][i]
#                     y = data['y_grid'][j]
#                     z = data['z_grid'][k]

#                     # lon, lat = ut.inverse_azi_equidist_proj(x, y, lon0, lat0)     
#                     lon, lat = ut.inverse_azi_equidist_proj(-x, y, lon0, lat0)     

#                     # Get velocity values (use None/empty if not available)
#                     vp = data['vp'][i, j, k] if data['vp'] is not None else ''
#                     vs = data['vs'][i, j, k] if data['vs'] is not None else ''
#                     vp_vs_ratio = data['vp_vs_ratio'][i, j, k] if data['vp_vs_ratio'] is not None else ''
                    
#                     # Write row
#                     writer.writerow({
#                         'lon': f"{lon:.3f}",
#                         'lat': f"{lat:.3f}",
#                         'x': f"{x:.3f}",
#                         'y': f"{y:.3f}", 
#                         'z': f"{z:.3f}",
#                         'vp': f"{vp:.3f}" if vp != '' else '',
#                         'vs': f"{vs:.3f}" if vs != '' else '',
#                         'vp_vs_ratio': f"{vp_vs_ratio:.4f}" if vp_vs_ratio != '' else ''
#                     })
    
#     print(f"CSV export complete: {output_filename}")
#     print(f"Columns: x, y, z" + 
#           (", vp" if data['vp'] is not None else "") +
#           (", vs" if data['vs'] is not None else "") + 
#           (", vp_vs_ratio" if data['vp_vs_ratio'] is not None else ""))

# def save_velocity_model(data, output_prefix):
#     """
#     Save the parsed velocity model to files.
    
#     Parameters:
#     -----------
#     data : dict
#         Dictionary returned by parse_simulps_velocity_file
#     output_prefix : str
#         Prefix for output filenames
#     """
    
#     # Save grids
#     np.savetxt(f"{output_prefix}_x_grid.txt", data['x_grid'], fmt='%.3f')
#     np.savetxt(f"{output_prefix}_y_grid.txt", data['y_grid'], fmt='%.3f')  
#     np.savetxt(f"{output_prefix}_z_grid.txt", data['z_grid'], fmt='%.3f')
    
#     # Save velocity models as binary
#     np.save(f"{output_prefix}_vp.npy", data['vp'])
#     np.save(f"{output_prefix}_vs.npy", data['vs'])
#     np.save(f"{output_prefix}_vp_vs_ratio.npy", data['vp_vs_ratio'])
    
#     # Save metadata
#     with open(f"{output_prefix}_metadata.txt", 'w') as f:
#         for key, value in data['metadata'].items():
#             f.write(f"{key}: {value}\n")
    
#     # Also save as CSV
#     export_to_csv(data, f"{output_prefix}_velocity_model.csv")
    
#     print(f"Saved velocity model with shape {data['vp'].shape if data['vp'] is not None else 'N/A'}")
#     print(f"X range: {data['x_grid'].min():.1f} to {data['x_grid'].max():.1f}")
#     print(f"Y range: {data['y_grid'].min():.1f} to {data['y_grid'].max():.1f}")
#     print(f"Z range: {data['z_grid'].min():.1f} to {data['z_grid'].max():.1f}")
#     if data['vp'] is not None:
#         print(f"Vp range: {data['vp'].min():.2f} to {data['vp'].max():.2f} km/s")
#     if data['vs'] is not None:
#         print(f"Vs range: {data['vs'].min():.2f} to {data['vs'].max():.2f} km/s")
#     if data['vp_vs_ratio'] is not None:
#         print(f"Vp/Vs ratio range: {data['vp_vs_ratio'].min():.3f} to {data['vp_vs_ratio'].max():.3f}")


# def get_velocity_at_location(data, x_target, y_target, z_target, parameter='vp'):
#     """
#     Get velocity value at a specific geographic location.
    
#     Parameters:
#     -----------
#     data : dict
#         Dictionary returned by parse_simulps_velocity_file
#     x_target, y_target, z_target : float
#         Target coordinates in km
#     parameter : str
#         Which parameter to retrieve: 'vp', 'vs', or 'vp_vs_ratio'
        
#     Returns:
#     --------
#     dict containing:
#         - 'value': interpolated value at target location
#         - 'nearest_indices': (i, j, k) indices of nearest grid point
#         - 'nearest_coords': (x, y, z) coordinates of nearest grid point
#         - 'nearest_value': value at nearest grid point
#     """
    
#     # Find nearest grid indices
#     x_idx = np.argmin(np.abs(data['x_grid'] - x_target))
#     y_idx = np.argmin(np.abs(data['y_grid'] - y_target))
#     z_idx = np.argmin(np.abs(data['z_grid'] - z_target))
    
#     # Get the appropriate data array
#     if parameter == 'vp':
#         velocity_array = data['vp']
#     elif parameter == 'vs':
#         velocity_array = data['vs']
#     elif parameter == 'vp_vs_ratio':
#         velocity_array = data['vp_vs_ratio']
#     else:
#         raise ValueError("parameter must be 'vp', 'vs', or 'vp_vs_ratio'")
    
#     # Get value at nearest grid point
#     nearest_value = velocity_array[x_idx, y_idx, z_idx]
    
#     # Get actual coordinates of nearest grid point
#     nearest_x = data['x_grid'][x_idx]
#     nearest_y = data['y_grid'][y_idx]
#     nearest_z = data['z_grid'][z_idx]
    
#     print(f"Target location: ({x_target}, {y_target}, {z_target}) km")
#     print(f"Nearest grid point: ({nearest_x}, {nearest_y}, {nearest_z}) km")
#     print(f"Array indices: [i={x_idx}, j={y_idx}, k={z_idx}]")
#     print(f"{parameter.upper()} value: {nearest_value:.3f}")
    
#     return {
#         'value': nearest_value,
#         'nearest_indices': (x_idx, y_idx, z_idx),
#         'nearest_coords': (nearest_x, nearest_y, nearest_z),
#         'nearest_value': nearest_value
#     }

# def verify_parsing_with_file_data(data, file_row_1indexed, file_col_1indexed, depth_layer_1indexed, expected_value, parameter='vp'):
#     """
#     Verify that a specific value from the original file is correctly parsed.
    
#     Parameters:
#     -----------
#     data : dict
#         Dictionary returned by parse_simulps_velocity_file
#     file_row_1indexed, file_col_1indexed : int
#         Row and column in the original file (1-indexed, as you would count them)
#     depth_layer_1indexed : int
#         Depth layer in the original file (1-indexed)
#     expected_value : float
#         The value you see in the original file
#     parameter : str
#         Which parameter to check: 'vp', 'vs', or 'vp_vs_ratio'
        
#     Returns:
#     --------
#     bool: True if parsing is correct
#     """
    
#     # Convert to 0-indexed
#     i = file_row_1indexed - 1      # X index
#     k = depth_layer_1indexed - 1   # Z index
    
#     # Y index mapping: file column j (0-indexed) -> array index (ny-1-j)
#     file_col_0indexed = file_col_1indexed - 1
#     j = data['metadata']['ny'] - 1 - file_col_0indexed
    
#     # Get the appropriate data array
#     if parameter == 'vp':
#         velocity_array = data['vp']
#     elif parameter == 'vs':
#         velocity_array = data['vs']
#     elif parameter == 'vp_vs_ratio':
#         velocity_array = data['vp_vs_ratio']
#     else:
#         raise ValueError("parameter must be 'vp', 'vs', or 'vp_vs_ratio'")
    
#     # Get parsed value
#     parsed_value = velocity_array[i, j, k]
    
#     # Get corresponding coordinates
#     x_coord = data['x_grid'][i]
#     y_coord = data['y_grid'][j]
#     z_coord = data['z_grid'][k]
    
#     print(f"\nVerification for {parameter.upper()}:")
#     print(f"File position: row {file_row_1indexed}, column {file_col_1indexed}, layer {depth_layer_1indexed}")
#     print(f"Array indices: [i={i}, j={j}, k={k}]")
#     print(f"Geographic coordinates: ({x_coord}, {y_coord}, {z_coord}) km")
#     print(f"Expected value: {expected_value}")
#     print(f"Parsed value: {parsed_value}")
#     print(f"Match: {'✓' if abs(parsed_value - expected_value) < 1e-6 else '✗'}")
    
#     return abs(parsed_value - expected_value) < 1e-6

# def plot_velocity_model(data, depth_index=0, parameter='vp', smooth=True, smooth_sigma=0.5):
#     """
#     Create a simple plot of a velocity slice.
    
#     Parameters:
#     -----------
#     data : dict
#         Dictionary returned by parse_simulps_velocity_file
#     depth_index : int
#         Index of depth layer to plot
#     parameter : str
#         Which parameter to plot: 'vp', 'vs', or 'vp_vs_ratio'
#     smooth : bool
#         Whether to apply Gaussian smoothing
#     smooth_sigma : float
#         Standard deviation for Gaussian smoothing (in grid units)
#     """
    
    
#     try:
#         import matplotlib.pyplot as plt
#         from scipy.ndimage import gaussian_filter
        
#         fig, ax = plt.subplots(figsize=(6, 4))
        
#         # Get the appropriate data array
#         if parameter == 'vp':
#             velocity_slice = data['vp'][:, :, depth_index]
#             label = 'Vp (km/s)'
#             title_param = 'P-wave velocity'
#             cmap = 'gist_rainbow_r'
#             cran = [4.0, 9.0]
#         elif parameter == 'vs':
#             velocity_slice = data['vs'][:, :, depth_index]
#             label = 'Vs (km/s)'
#             title_param = 'S-wave velocity'
#             cmap = 'gist_rainbow_r'
#             cran = [1.5, 6]
#         elif parameter == 'vp_vs_ratio':
#             velocity_slice = data['vp_vs_ratio'][:, :, depth_index]
#             label = 'Vp/Vs ratio'
#             title_param = 'Vp/Vs ratio'
#             cmap = 'gist_rainbow'
#             cran = [1.6, 2.0]
#         else:
#             raise ValueError("parameter must be 'vp', 'vs', or 'vp_vs_ratio'")
        
#         depth_km = data['z_grid'][depth_index]
 
#          # Apply smoothing if requested
#         if smooth:
#             velocity_slice = gaussian_filter(velocity_slice, sigma=smooth_sigma)
       
#             im = ax.imshow(velocity_slice, 
#                         extent=[data['x_grid'].min(), data['x_grid'].max(),
#                                 data['y_grid'].min(), data['y_grid'].max()],
#                         aspect='equal', cmap=cmap, vmin=cran[0], vmax=cran[1],
#                         origin='lower', interpolation='bilinear')
            
#         else:
#             im = ax.imshow(velocity_slice, 
#                         extent=[data['x_grid'].min(), data['x_grid'].max(),
#                                 data['y_grid'].min(), data['y_grid'].max()],
#                         aspect='equal', cmap=cmap, vmin=cran[0], vmax=cran[1],
#                         origin='lower')
        
#         ax.set_xlabel('X (km)')
#         ax.set_ylabel('Y (km)')
#         ax.set_title(f'{title_param} at depth {depth_km} km' + (' (smoothed)' if smooth else ''))
        
#         plt.colorbar(im, ax=ax, label=label)
#         plt.tight_layout()
#         plt.show()
        
#     except ImportError as e:
#         if 'scipy' in str(e):
#             print("SciPy not available for smoothing. Plotting without smoothing.")
#             # Fallback to plotting without smoothing
#             plot_velocity_model(data, depth_index, parameter, smooth=False)
#         else:
#             print("Matplotlib not available for plotting")

# # Example usage:
# if __name__ == "__main__":
#     # Parse the file
#     # data = parse_simulps_velocity_file("your_simulps_file.txt")
    
#     # Save to files
#     # save_velocity_model(data, "parsed_model")
    
#     # Plot different parameters at various depths
#     # plot_velocity_model(data, depth_index=0, parameter='vp')     # P-wave velocity
#     # plot_velocity_model(data, depth_index=0, parameter='vs')     # S-wave velocity  
#     # plot_velocity_model(data, depth_index=0, parameter='vp_vs_ratio')  # Vp/Vs ratio
    
#     print("SIMULPS parser ready. Use parse_simulps_velocity_file('filename') to parse your file.")
#     print("The parser will extract Vp, Vs, and Vp/Vs ratio data in that order.")

In [ ]:
import numpy as np
import re

def detect_simulps_content(filename):
    """
    Detect what velocity data is contained in a SIMULPS file based on line count.
    
    Parameters:
    -----------
    filename : str
        Path to the SIMULPS velocity file
        
    Returns:
    --------
    dict containing:
        - 'content_type': str - 'vp_only', 'vp_vs', or 'vp_vs_ratio'
        - 'has_vp': bool
        - 'has_vs': bool  
        - 'has_vp_vs_ratio': bool
        - 'expected_data_lines': int
        - 'actual_data_lines': int
        - 'nx': int, 'ny': int, 'nz': int
    """
    
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    # Parse header to get dimensions
    header = lines[0].strip().split()
    nx = int(header[1])
    ny = int(header[2]) 
    nz = int(header[3])
    
    # Calculate expected lines
    header_lines = 5  # header + 3 coordinate lines + "0 0 0" line
    total_lines = len(lines)
    data_lines = total_lines - header_lines
    
    lines_per_layer = nx  # Each depth layer has nx rows
    lines_for_one_dataset = lines_per_layer * nz  # One complete 3D dataset
    
    print(f"File analysis:")
    print(f"Grid dimensions: {nx} x {ny} x {nz}")
    print(f"Total lines in file: {total_lines}")
    print(f"Header lines: {header_lines}")
    print(f"Data lines: {data_lines}")
    print(f"Lines per complete dataset: {lines_for_one_dataset}")
    
    # Determine content based on data lines
    if data_lines == lines_for_one_dataset:
        content_type = 'vp_only'
        has_vp, has_vs, has_ratio = True, False, False
        print("Content detected: Vp only")
        
    elif data_lines == 2 * lines_for_one_dataset:
        content_type = 'vp_vs'
        has_vp, has_vs, has_ratio = True, True, False
        print("Content detected: Vp and Vs")
        
    elif data_lines == 3 * lines_for_one_dataset:
        content_type = 'vp_vs_ratio'
        has_vp, has_vs, has_ratio = True, True, True
        print("Content detected: Vp, Vs, and Vp/Vs ratio")
        
    else:
        print(f"WARNING: Unexpected number of data lines!")
        print(f"Expected: {lines_for_one_dataset} (Vp only), {2*lines_for_one_dataset} (Vp+Vs), or {3*lines_for_one_dataset} (Vp+Vs+ratio)")
        print(f"Found: {data_lines}")
        content_type = 'unknown'
        has_vp, has_vs, has_ratio = False, False, False
    
    return {
        'content_type': content_type,
        'has_vp': has_vp,
        'has_vs': has_vs,
        'has_vp_vs_ratio': has_ratio,
        'expected_data_lines': lines_for_one_dataset,
        'actual_data_lines': data_lines,
        'nx': nx,
        'ny': ny,
        'nz': nz
    }

def parse_simulps_velocity_file(filename):
    """
    Parse a SIMULPS velocity model file and extract grids and velocity data.
    Automatically detects whether file contains Vp only, Vp+Vs, or Vp+Vs+ratio.
    
    Parameters:
    -----------
    filename : str
        Path to the SIMULPS velocity file
        
    Returns:
    --------
    dict containing:
        - 'x_grid': numpy array of x coordinates
        - 'y_grid': numpy array of y coordinates  
        - 'z_grid': numpy array of z coordinates (depths)
        - 'vp': numpy array of shape (nx, ny, nz) with P-wave velocity values (if present)
        - 'vs': numpy array of shape (nx, ny, nz) with S-wave velocity values (if present)
        - 'vp_vs_ratio': numpy array of shape (nx, ny, nz) with Vp/Vs ratio values (if present)
        - 'metadata': dict with file metadata
        - 'content_info': dict with content detection results
    """
    
    # First detect what's in the file
    content_info = detect_simulps_content(filename)
    
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    # Parse header line
    header = lines[0].strip().split()
    version = float(header[0])
    nx = int(header[1])  # number of x nodes
    ny = int(header[2])  # number of y nodes  
    nz = int(header[3])  # number of z layers
    # wave_type = int(header[4])  # 1=P-wave, 2=S-wave
    # status = header[5] if len(header) > 5 else ""
    # timestamp = " ".join(header[6:]) if len(header) > 6 else ""
    
    print(f"Parsing SIMULPS file: {nx}x{ny}x{nz} grid")
    # print(f"Wave type: {'P-wave' if wave_type == 1 else 'S-wave' if wave_type == 2 else 'Unknown'}")
    print(f"File contains: Vp, Vs, and Vp/Vs ratio data")
    
    # Parse coordinate arrays
    x_coords = np.array([float(x) for x in lines[1].strip().split()])
    y_coords = np.array([float(y) for y in lines[2].strip().split()])
    z_coords = np.array([float(z) for z in lines[3].strip().split()])
    
    # Verify coordinate array lengths
    if len(x_coords) != nx:
        raise ValueError(f"X coordinates length ({len(x_coords)}) doesn't match nx ({nx})")
    if len(y_coords) != ny:
        raise ValueError(f"Y coordinates length ({len(y_coords)}) doesn't match ny ({ny})")
    if len(z_coords) != nz:
        raise ValueError(f"Z coordinates length ({len(z_coords)}) doesn't match nz ({nz})")
    
    # Skip the "0 0 0" line (line 4, index 3)
    data_start_line = 5
    
    # Initialize velocity arrays for Vp, Vs, and Vp/Vs ratio
    vp = np.zeros((nx, ny, nz))
    vs = np.zeros((nx, ny, nz))
    vp_vs_ratio = np.zeros((nx, ny, nz))
    
    # Parse velocity data based on detected content
    current_line = data_start_line
    
    # Parse Vp data (always present)
    if content_info['has_vp']:
        print("Reading Vp data...")
        for k in range(nz):  # For each depth layer
            print(f"  Vp depth layer {k+1}/{nz} (z = {z_coords[k]} km)")
            
            # Read nx rows for this depth layer
            for i in range(nx):  # For each y row (top to bottom)
                if current_line >= len(lines):
                    raise ValueError(f"Unexpected end of file at Vp depth layer {k+1}, row {i+1}")
                    
                line = lines[current_line].strip()
                current_line += 1
                
                # Parse the line
                row_values = parse_velocity_line(line, ny)
                
                if len(row_values) != ny:
                    raise ValueError(f"Vp row {i+1} in depth layer {k+1} has {len(row_values)} values, expected {ny}")
                
                # File row i corresponds to y_coords[i] (top to bottom)
                # File column j corresponds to x_coords[j] (left to right)
                # So: vp[j, i, k] = row_values[j] where j=X index, i=Y index
                for j in range(ny):
                    vp[j, i, k] = row_values[j]
    
    # Parse Vs data (if present)
    if content_info['has_vs']:
        print("Reading Vs data...")
        for k in range(nz):  # For each depth layer
            print(f"  Vs depth layer {k+1}/{nz} (z = {z_coords[k]} km)")
            
            # Read nx rows for this depth layer
            for i in range(nx):  # For each y row (top to bottom)
                if current_line >= len(lines):
                    raise ValueError(f"Unexpected end of file at Vs depth layer {k+1}, row {i+1}")
                    
                line = lines[current_line].strip()
                current_line += 1
                
                # Parse the line
                row_values = parse_velocity_line(line, ny)
                
                if len(row_values) != ny:
                    raise ValueError(f"Vs row {i+1} in depth layer {k+1} has {len(row_values)} values, expected {ny}")
                
                # File row i corresponds to y_coords[i] (top to bottom)
                # File column j corresponds to x_coords[j] (left to right)
                # So: vs[j, i, k] = row_values[j] where j=X index, i=Y index
                for j in range(ny):
                    vs[j, i, k] = row_values[j]
    
    # Parse Vp/Vs ratio data (if present)
    if content_info['has_vp_vs_ratio']:
        print("Reading Vp/Vs ratio data...")
        for k in range(nz):  # For each depth layer
            print(f"  Vp/Vs depth layer {k+1}/{nz} (z = {z_coords[k]} km)")
            
            # Read nx rows for this depth layer
            for i in range(nx):  # For each y row (top to bottom)
                if current_line >= len(lines):
                    raise ValueError(f"Unexpected end of file at Vp/Vs depth layer {k+1}, row {i+1}")
                    
                line = lines[current_line].strip()
                current_line += 1
                
                # Parse the line
                row_values = parse_velocity_line(line, ny)
                
                if len(row_values) != ny:
                    raise ValueError(f"Vp/Vs row {i+1} in depth layer {k+1} has {len(row_values)} values, expected {ny}")
                
                # File row i corresponds to y_coords[i] (top to bottom)
                # File column j corresponds to x_coords[j] (left to right)
                # So: vp_vs_ratio[j, i, k] = row_values[j] where j=X index, i=Y index
                for j in range(ny):
                    vp_vs_ratio[j, i, k] = row_values[j]
    
    # Create metadata dictionary
    metadata = {
        'version': version,
        'nx': nx,
        'ny': ny, 
        'nz': nz,
        # 'wave_type': wave_type,
        # 'status': status,
        # 'timestamp': timestamp
    }
    
    return {
        'x_grid': x_coords,
        'y_grid': y_coords,
        'z_grid': z_coords,
        'vp': vp,
        'vs': vs,
        'vp_vs_ratio': vp_vs_ratio,
        'metadata': metadata,
        'content_info': content_info
    }

def parse_velocity_line(line, expected_count):
    """
    Parse a line of velocity values that may be space-separated or concatenated.
    
    Parameters:
    -----------
    line : str
        Line containing velocity values
    expected_count : int
        Expected number of values in the line
        
    Returns:
    --------
    list of float values
    """
    
    # First try space-separated parsing
    if ' ' in line:
        values = [float(x) for x in line.split()]
        if len(values) == expected_count:
            return values
    
    # If space-separated doesn't work or gives wrong count, try pattern matching
    # Look for patterns like "3.00", "10.30", etc.
    
    # Try different decimal patterns
    patterns = [
        r'\d+\.\d{2}',  # Format like 3.00, 10.30
        r'\d+\.\d{1}',  # Format like 3.0, 10.3  
        r'\d+\.\d{3}',  # Format like 3.000, 10.300
        r'\d+\.\d+',    # Any decimal format
        r'\d+'          # Integer format
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, line)
        if len(matches) == expected_count:
            return [float(x) for x in matches]
    
    # If pattern matching fails, try fixed-width parsing
    # Assume each value takes the same number of characters
    line_clean = line.replace(' ', '')
    if len(line_clean) % expected_count == 0:
        width = len(line_clean) // expected_count
        values = []
        for i in range(expected_count):
            start_idx = i * width
            end_idx = start_idx + width
            value_str = line_clean[start_idx:end_idx]
            try:
                values.append(float(value_str))
            except ValueError:
                break
        if len(values) == expected_count:
            return values
    
    # If all else fails, raise an error
    raise ValueError(f"Could not parse velocity line: '{line}' (expected {expected_count} values)")

def export_to_csv(data, output_filename):
    """
    Export the parsed velocity model to a CSV file with columns: x, y, z, vp, vs, vp_vs_ratio
    
    Parameters:
    -----------
    data : dict
        Dictionary returned by parse_simulps_velocity_file
    output_filename : str
        Output CSV filename
    """
    
    import csv
    import utils as ut    
    
    nx, ny, nz = data['metadata']['nx'], data['metadata']['ny'], data['metadata']['nz']
    
    print(f"Exporting velocity model to CSV: {output_filename}")
    print(f"Grid dimensions: {nx} x {ny} x {nz} = {nx*ny*nz} points")
    
    with open(output_filename, 'w', newline='') as csvfile:
        # fieldnames = ['x', 'y', 'z', 'vp', 'vs', 'vp_vs_ratio']
        fieldnames = ['lon', 'lat', 'x', 'y', 'z', 'vp', 'vs', 'vp_vs_ratio']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        # Write header
        writer.writeheader()
        
        # define the centroid of relative coordinates,
        # found from the fist line in "DeShon_2006GJI/NICOYA_MODEL/eq10ps.f3dps.overdamp.stns.out"
        lon0, lat0 = -(84+42.00/60), 9+12.00/60 
        
        # Write data for each grid point
        # Array indexing: velocity[j, i, k] where j=X index, i=Y index, k=Z index
        for j in range(ny):  # X index
            for i in range(nx):  # Y index
                for k in range(nz):  # Z index
                    # Get coordinates
                    x = data['x_grid'][j]  # X coordinate from j index
                    y = data['y_grid'][i]  # Y coordinate from i index
                    z = data['z_grid'][k]
                    
                    ### This is the key!
                    # X coordinates in the file represent longitude offsets from center:
                    # Negative X values → East of center (84°42.00'W)
                    # Positive X values → West of center (84°42.00'W)
                    # Y coordinates in the file represent latitude offsets from center:
                    # Negative Y values → South of center (9°12.00'N)
                    # Positive Y values → North of center (9°12.00'N)

                    # Based on above, should reverse x for projection back to lon and lat
                    lon, lat = ut.inverse_azi_equidist_proj(-x, y, lon0, lat0)     

                    # Get velocity values (use None/empty if not available)
                    vp = data['vp'][j, i, k] if data['vp'] is not None else ''
                    vs = data['vs'][j, i, k] if data['vs'] is not None else ''
                    vp_vs_ratio = data['vp_vs_ratio'][j, i, k] if data['vp_vs_ratio'] is not None else ''
                    
                    # Write row
                    writer.writerow({
                        'lon': f"{lon:.3f}",
                        'lat': f"{lat:.3f}",
                        'x': f"{x:.3f}",
                        'y': f"{y:.3f}", 
                        'z': f"{z:.3f}",
                        'vp': f"{vp:.3f}" if vp != '' else '',
                        'vs': f"{vs:.3f}" if vs != '' else '',
                        'vp_vs_ratio': f"{vp_vs_ratio:.4f}" if vp_vs_ratio != '' else ''
                    })
    
    print(f"CSV export complete: {output_filename}")
    print(f"Columns: x, y, z" + 
          (", vp" if data['vp'] is not None else "") +
          (", vs" if data['vs'] is not None else "") + 
          (", vp_vs_ratio" if data['vp_vs_ratio'] is not None else ""))

def save_velocity_model(data, output_prefix):
    """
    Save the parsed velocity model to files.
    
    Parameters:
    -----------
    data : dict
        Dictionary returned by parse_simulps_velocity_file
    output_prefix : str
        Prefix for output filenames
    """
    
    # Save grids
    np.savetxt(f"{output_prefix}_x_grid.txt", data['x_grid'], fmt='%.3f')
    np.savetxt(f"{output_prefix}_y_grid.txt", data['y_grid'], fmt='%.3f')  
    np.savetxt(f"{output_prefix}_z_grid.txt", data['z_grid'], fmt='%.3f')
    
    # Save velocity models as binary
    np.save(f"{output_prefix}_vp.npy", data['vp'])
    np.save(f"{output_prefix}_vs.npy", data['vs'])
    np.save(f"{output_prefix}_vp_vs_ratio.npy", data['vp_vs_ratio'])
    
    # Save metadata
    with open(f"{output_prefix}_metadata.txt", 'w') as f:
        for key, value in data['metadata'].items():
            f.write(f"{key}: {value}\n")
    
    # Also save as CSV
    export_to_csv(data, f"{output_prefix}_velocity_model.csv")
    
    print(f"Saved velocity model with shape {data['vp'].shape if data['vp'] is not None else 'N/A'}")
    print(f"X range: {data['x_grid'].min():.1f} to {data['x_grid'].max():.1f}")
    print(f"Y range: {data['y_grid'].min():.1f} to {data['y_grid'].max():.1f}")
    print(f"Z range: {data['z_grid'].min():.1f} to {data['z_grid'].max():.1f}")
    if data['vp'] is not None:
        print(f"Vp range: {data['vp'].min():.2f} to {data['vp'].max():.2f} km/s")
    if data['vs'] is not None:
        print(f"Vs range: {data['vs'].min():.2f} to {data['vs'].max():.2f} km/s")
    if data['vp_vs_ratio'] is not None:
        print(f"Vp/Vs ratio range: {data['vp_vs_ratio'].min():.3f} to {data['vp_vs_ratio'].max():.3f}")

def get_velocity_at_location(data, x_target, y_target, z_target, parameter='vp'):
    """
    Get velocity value at a specific geographic location.
    
    Parameters:
    -----------
    data : dict
        Dictionary returned by parse_simulps_velocity_file
    x_target, y_target, z_target : float
        Target coordinates in km
    parameter : str
        Which parameter to retrieve: 'vp', 'vs', or 'vp_vs_ratio'
        
    Returns:
    --------
    dict containing:
        - 'value': interpolated value at target location
        - 'nearest_indices': (i, j, k) indices of nearest grid point
        - 'nearest_coords': (x, y, z) coordinates of nearest grid point
        - 'nearest_value': value at nearest grid point
    """
    
    # Find nearest grid indices
    x_idx = np.argmin(np.abs(data['x_grid'] - x_target))
    y_idx = np.argmin(np.abs(data['y_grid'] - y_target))
    z_idx = np.argmin(np.abs(data['z_grid'] - z_target))
    
    # Get the appropriate data array
    if parameter == 'vp':
        velocity_array = data['vp']
    elif parameter == 'vs':
        velocity_array = data['vs']
    elif parameter == 'vp_vs_ratio':
        velocity_array = data['vp_vs_ratio']
    else:
        raise ValueError("parameter must be 'vp', 'vs', or 'vp_vs_ratio'")
    
    # Get value at nearest grid point
    nearest_value = velocity_array[x_idx, y_idx, z_idx]
    
    # Get actual coordinates of nearest grid point
    nearest_x = data['x_grid'][x_idx]
    nearest_y = data['y_grid'][y_idx]
    nearest_z = data['z_grid'][z_idx]
    
    print(f"Target location: ({x_target}, {y_target}, {z_target}) km")
    print(f"Nearest grid point: ({nearest_x}, {nearest_y}, {nearest_z}) km")
    print(f"Array indices: [i={x_idx}, j={y_idx}, k={z_idx}]")
    print(f"{parameter.upper()} value: {nearest_value:.3f}")
    
    return {
        'value': nearest_value,
        'nearest_indices': (x_idx, y_idx, z_idx),
        'nearest_coords': (nearest_x, nearest_y, nearest_z),
        'nearest_value': nearest_value
    }

def verify_parsing_with_file_data(data, file_row_1indexed, file_col_1indexed, depth_layer_1indexed, expected_value, parameter='vp'):
    """
    Verify that a specific value from the original file is correctly parsed.
    
    Parameters:
    -----------
    data : dict
        Dictionary returned by parse_simulps_velocity_file
    file_row_1indexed, file_col_1indexed : int
        Row and column in the original file (1-indexed, as you would count them)
    depth_layer_1indexed : int
        Depth layer in the original file (1-indexed)
    expected_value : float
        The value you see in the original file
    parameter : str
        Which parameter to check: 'vp', 'vs', or 'vp_vs_ratio'
        
    Returns:
    --------
    bool: True if parsing is correct
    """
    
    # Convert to 0-indexed
    i = file_row_1indexed - 1      # Y index (file row -> Y coordinate)
    j = file_col_1indexed - 1      # X index (file column -> X coordinate)
    k = depth_layer_1indexed - 1   # Z index
    
    # Get the appropriate data array
    if parameter == 'vp':
        velocity_array = data['vp']
    elif parameter == 'vs':
        velocity_array = data['vs']
    elif parameter == 'vp_vs_ratio':
        velocity_array = data['vp_vs_ratio']
    else:
        raise ValueError("parameter must be 'vp', 'vs', or 'vp_vs_ratio'")
    
    # Get parsed value: velocity_array[j, i, k] where j=X index, i=Y index
    parsed_value = velocity_array[j, i, k]
    
    # Get corresponding coordinates
    x_coord = data['x_grid'][j]  # X coordinate from column index
    y_coord = data['y_grid'][i]  # Y coordinate from row index
    z_coord = data['z_grid'][k]
    
    print(f"\nVerification for {parameter.upper()}:")
    print(f"File position: row {file_row_1indexed}, column {file_col_1indexed}, layer {depth_layer_1indexed}")
    print(f"Array indices: [j={j}(X), i={i}(Y), k={k}(Z)]")
    print(f"Geographic coordinates: ({x_coord}, {y_coord}, {z_coord}) km")
    print(f"Expected value: {expected_value}")
    print(f"Parsed value: {parsed_value}")
    print(f"Match: {'✓' if abs(parsed_value - expected_value) < 1e-6 else '✗'}")
    
    return abs(parsed_value - expected_value) < 1e-6

def plot_velocity_model(data, depth_index=0, parameter='vp', smooth=True, smooth_sigma=0.5):
    """
    Create a simple plot of a velocity slice.
    
    Parameters:
    -----------
    data : dict
        Dictionary returned by parse_simulps_velocity_file
    depth_index : int
        Index of depth layer to plot
    parameter : str
        Which parameter to plot: 'vp', 'vs', or 'vp_vs_ratio'
    smooth : bool
        Whether to apply Gaussian smoothing
    smooth_sigma : float
        Standard deviation for Gaussian smoothing (in grid units)
    """
    
    try:
        import matplotlib.pyplot as plt
        from scipy.ndimage import gaussian_filter
        
        fig, ax = plt.subplots(figsize=(6, 5))
        
        # Get the appropriate data array
        if parameter == 'vp':
            velocity_slice = data['vp'][:, :, depth_index]
            label = 'Vp (km/s)'
            title_param = 'P-wave velocity'
            cran = [4.0, 9.0]  # color range
            cmap='gist_rainbow_r'
        elif parameter == 'vs':
            velocity_slice = data['vs'][:, :, depth_index]
            label = 'Vs (km/s)'
            title_param = 'S-wave velocity'
            cran = [3.0, 6.0]  # color range
            cmap='gist_rainbow_r'
        elif parameter == 'vp_vs_ratio':
            velocity_slice = data['vp_vs_ratio'][:, :, depth_index]
            label = 'Vp/Vs ratio'
            title_param = 'Vp/Vs ratio'
            cran = [1.6, 2.0]  # color range
            cmap='gist_rainbow'
        else:
            raise ValueError("parameter must be 'vp', 'vs', or 'vp_vs_ratio'")
        
        depth_km = data['z_grid'][depth_index]

        # Apply smoothing if requested
        if smooth:
            velocity_slice = gaussian_filter(velocity_slice, sigma=smooth_sigma)
        
            im = ax.imshow(velocity_slice, 
                        extent=[data['x_grid'].min(), data['x_grid'].max(),
                                data['y_grid'].min(), data['y_grid'].max()],
                        aspect='equal', cmap=cmap, vmin=cran[0], vmax=cran[1],
                        origin='lower', interpolation='bilinear')

        else: 
            im = ax.imshow(velocity_slice, 
                    extent=[data['x_grid'].min(), data['x_grid'].max(),
                            data['y_grid'].min(), data['y_grid'].max()],
                    aspect='equal', cmap=cmap, vmin=cran[0], vmax=cran[1],
                    origin='lower')
                       
        ax.set_xlabel('X (km)')
        ax.set_ylabel('Y (km)')
        ax.set_title(f'{title_param} at depth {depth_km} km' + (' (smoothed)' if smooth else ''))
        
        plt.colorbar(im, ax=ax, label=label)
        plt.tight_layout()
        plt.show()
        
    except ImportError as e:
        if 'scipy' in str(e):
            print("SciPy not available for smoothing. Plotting without smoothing.")
            # Fallback to plotting without smoothing
            plot_velocity_model(data, depth_index, parameter, smooth=False)
        else:
            print("Matplotlib not available for plotting")

# Example usage:
if __name__ == "__main__":
    # Parse the file and export to CSV
    # data = parse_simulps_velocity_file("your_simulps_file.txt")
    # export_to_csv(data, "velocity_model.csv")
    
    # Or save everything including CSV
    # save_velocity_model(data, "parsed_model")  # This now includes CSV export
    
    # Plot different parameters at various depths
    # plot_velocity_model(data, depth_index=0, parameter='vp')     # P-wave velocity
    # plot_velocity_model(data, depth_index=0, parameter='vs')     # S-wave velocity  
    # plot_velocity_model(data, depth_index=0, parameter='vp_vs_ratio')  # Vp/Vs ratio
    
    print("SIMULPS parser ready. Use parse_simulps_velocity_file('filename') to parse your file.")
    print("Use export_to_csv(data, 'output.csv') to export to CSV format.")
    print("The parser will extract Vp, Vs, and Vp/Vs ratio data based on file content.")

In [ ]:
datadir = "/home/staff/chao/SSEinv/Nicoya/DeShon_2006GJI/"
modelFile = "NICOYA_MODEL/eq10ps.f3dps.overdamp.mod.out"

# Check what's in your file first
content_info = detect_simulps_content(datadir + modelFile)
print(content_info['content_type'])  # 'vp_only', 'vp_vs', or 'vp_vs_ratio'

# Parse your complete SIMULPS file
data = parse_simulps_velocity_file(datadir + modelFile)

# Access all three datasets
vp = data['vp']              # P-wave velocities (nx, ny, nz)
vs = data['vs']              # S-wave velocities (nx, ny, nz)  
vp_vs_ratio = data['vp_vs_ratio']  # Vp/Vs ratios (nx, ny, nz)

# Export to CSV
export_to_csv(data, "DeShon2006_3Dmodel.csv")

# Or use save_velocity_model which now includes CSV export, 
# This creates: my_model_velocity_model.csv + other files
# save_velocity_model(data, "DeShon2006")  


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

veldir = "/home/staff/chao/SSEinv/Nicoya/DeShon_2006GJI/"
vel3dfile = "DeShon2006_3Dmodel.csv"
vel3d = pd.read_csv(veldir + vel3dfile, sep=",")
print(vel3d.head())

def plot_velocity(vel3d, depth_km=9, parameter='vp'):

    vel3d_slice = vel3d[(vel3d['z'] == depth_km)].reset_index(drop=True)

    if parameter == 'vp':
        slice = vel3d_slice['vp']
        label = 'Vp (km/s)'
        title_param = 'P-wave velocity'
        cran = [4.0, 9.0]  # color range
        cmap='gist_rainbow_r'
    elif parameter == 'vs':
        slice = vel3d_slice['vs']
        label = 'Vs (km/s)'
        title_param = 'S-wave velocity'
        cran = [3.0, 6.0]  # color range
        cmap='gist_rainbow_r'
    elif parameter == 'vp_vs_ratio':
        slice = vel3d_slice['vp_vs_ratio']
        label = 'Vp/Vs ratio'
        title_param = 'Vp/Vs ratio'
        cran = [1.6, 2.0]  # color range
        cmap='gist_rainbow'
    else:
        raise ValueError("parameter must be 'vp', 'vs', or 'vp_vs_ratio'")


    plt.figure(figsize=(6, 5))
    sc = plt.scatter(
        vel3d_slice['lon'], vel3d_slice['lat'],
        c=slice,           # use vp column for color
        cmap=cmap,
        s=70, marker='s',
        vmin=cran[0], vmax=cran[1]
    )
    plt.colorbar(sc, label=label)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title(f'{title_param} at depth {depth_km} km')
    # plt.axis('equal')
    plt.xlim([-87, -84.5])
    plt.ylim([9, 11.5])
    plt.show()

plot_velocity(vel3d, depth_km=9, parameter='vp')
# plot_velocity(vel3d, depth_km=20, parameter='vp')
# plot_velocity(vel3d, depth_km=35, parameter='vp')
# plot_velocity(vel3d, depth_km=50, parameter='vp')

# plot_velocity(vel3d, depth_km=9, parameter='vp_vs_ratio')
# plot_velocity(vel3d, depth_km=20, parameter='vp_vs_ratio')
# plot_velocity(vel3d, depth_km=35, parameter='vp_vs_ratio')
# plot_velocity(vel3d, depth_km=50, parameter='vp_vs_ratio')


In [ ]:
# Plot different parameters
plot_velocity_model(data, depth_index=3, parameter='vp', smooth=False)
# plot_velocity_model(data, depth_index=6, parameter='vp', smooth=False)
# plot_velocity_model(data, depth_index=9, parameter='vp', smooth=False)
# plot_velocity_model(data, depth_index=11, parameter='vp', smooth=False)

# plot_velocity_model(data, depth_index=5, parameter='vs')

# plot_velocity_model(data, depth_index=3, parameter='vp_vs_ratio')
# plot_velocity_model(data, depth_index=6, parameter='vp_vs_ratio')
# plot_velocity_model(data, depth_index=9, parameter='vp_vs_ratio')
# plot_velocity_model(data, depth_index=11, parameter='vp_vs_ratio')


In [ ]:
# Check Vp at location (40, 80, 20) km
result = get_velocity_at_location(data, x_target=40, y_target=80, z_target=20, parameter='vp')

In [ ]:
# Verify the 8.23 value you mentioned earlier
# File row 5, column 14, assuming it's in layer 7 of Vp data
verify_parsing_with_file_data(data, 
                             file_row_1indexed=5, 
                             file_col_1indexed=14, 
                             depth_layer_1indexed=7,  # which depth layer? 
                             expected_value=8.23, 
                             parameter='vp')

In [ ]:
# If you know the exact indices, check directly:
i, j, k = 4, 5, 0  # example indices
print(f"Vp at array[{i},{j},{k}] = {data['vp'][i,j,k]}")
print(f"Coordinates: ({data['x_grid'][i]}, {data['y_grid'][j]}, {data['z_grid'][k]})")

In [ ]:
refmodelFile = "NICOYA_MODEL/startingmod"

# Check what's in your file first
content_info = detect_simulps_content(datadir + refmodelFile)
print(content_info['content_type'])  # 'vp_only', 'vp_vs', or 'vp_vs_ratio'

# Parse your complete SIMULPS file
data_ref = parse_simulps_velocity_file(datadir + refmodelFile)

# Export to CSV
export_to_csv(data_ref, "DeShon2006_1DVp.csv")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

datadir = "/home/staff/chao/SSEinv/Nicoya/DeShon_2006GJI/"
nodefile = "nicoya.pi.1020304050"

# === Read the file ===
# Replace 'your_file.txt' with the actual path
df = pd.read_csv(datadir + nodefile, delim_whitespace=True, header=None, names=['lon', 'lat', 'depth'])

# === Simple scatter plot ===
plt.figure(figsize=(6, 6))
plt.scatter(df['lon'], df['lat'], c='black', s=10, marker='o')  # solid circles
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Scatter of Longitude vs Latitude')
plt.gca().set_aspect('equal')
plt.grid(True)
plt.show()

In [ ]:
import utils as ut

# define the centroid of relative coordinates,
# found from the fist line in "DeShon_2006GJI/NICOYA_MODEL/eq10ps.f3dps.overdamp.stns.out"
lon0, lat0 = -(84+42.00/60), 9+12.00/60 

lon, lat = -(84+57.38/60), 9+55.75/60 

# convert to relative locations in km
### This is the key!
# X coordinates in the file represent longitude offsets from center:
# Negative X values → East of center (84°42.00'W)
# Positive X values → West of center (84°42.00'W)
# Y coordinates in the file represent latitude offsets from center:
# Negative Y values → South of center (9°12.00'N)
# Positive Y values → North of center (9°12.00'N)

x, y = ut.azi_equidist_proj(lon, lat, lon0, lat0)
print(-x, y)

In [ ]:
# read in the 1D reference velocity model from Table 1 in DeShon et al. (2006)
# This has the same depth layer, so we use ref model for locations excluding the 3D extent
import pandas as pd
datadir = "/home/staff/chao/SSEinv/Nicoya/DeShon_2006GJI/"
ref_model1D = "DeShon2006_1Dmodel.csv"
datadir + ref_model1D
df = pd.read_csv(datadir + ref_model1D, sep=r'\s+',skiprows=1, \
                 names=['dep', 'vp', 'vs', 'vp_vs_ratio'])

print(df.head())


In [ ]:
# read in the 1D reference velocity model from Table 1 in DeShon & Schwartz (2004) GRL
# This has diff depth layers, but offers reference densities
import pandas as pd
datadir = "/home/staff/chao/SSEinv/Nicoya/DeShon_2006GJI/"
ref_model1D = "DeShon2004_1Dmodel.csv"
datadir + ref_model1D
df = pd.read_csv(datadir + ref_model1D, sep=r'\s+',skiprows=1, \
                 names=['dep', 'vp', 'vp_vs_ratio', 'den'])

print(df.head())